# ETL Demo

This notebook performs a demo ETL transformation.

In [ ]:
# Widget setup for catalog and schema (safe defaults)
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "default")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
print(f"➡️ Using catalog={catalog}, schema={schema}")

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Resolve catalog/schema safely
env_catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog")
env_schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema")
catalog = env_catalog or catalog
schema = env_schema or schema
if catalog in ("hive_metastore", None):
    raise ValueError(f"❌ Catalog resolved to {catalog}. Check databricks.yml target overrides!")
print(f"➡️ Final resolved catalog={catalog}, schema={schema}")

input_table = f"{catalog}.{schema}.demo_sales"
output_table = f"{catalog}.{schema}.etl_demo_output"

print(f"Reading input table: {input_table}")
df = spark.read.table(input_table)

print("Adding amount_with_tax column...")
df_transformed = df.withColumn("amount_with_tax", F.col("amount") * 1.2)

print(f"Writing transformed table: {output_table}")
df_transformed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table)
print(f"✅ ETL demo table created at {output_table}")

In [ ]:
display(spark.table(output_table))